# Thông tin sinh viên

*   MSSV: xxxxxxx
*   Họ và tên: xxxxxxx



#Bài tập 3

#Kết nối/ Mount đến Google Drive

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


#Khai báo thư viện

In [4]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt

from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression

from sklearn.model_selection import GridSearchCV, RandomizedSearchCV

from sklearn.metrics import f1_score, accuracy_score, classification_report

# Đọc dữ liệu

In [5]:
# Hàm đọc dữ liệu từ thư mục folder
def load_images_from_folder(folder):
    images = []
    labels = []
    class_names = sorted(os.listdir(folder))

    for label, class_name in enumerate(class_names):
        class_folder = os.path.join(folder, class_name)
        if os.path.isdir(class_folder):
            for filename in os.listdir(class_folder):
                img_path = os.path.join(class_folder, filename)
                img = cv2.imread(img_path)
                if img is not None:
                    images.append(img)
                    labels.append(label)

    return images, labels, class_names


In [6]:
# Đọc dữ liệu từ thư mục Dataset
train_folder  = "/content/drive/MyDrive/CS231/Dataset/HoaVietNam/train"
test_folder   = "/content/drive/MyDrive/CS231/Dataset/HoaVietNam/test"

train_images, train_labels, class_names = load_images_from_folder(train_folder)
test_images, test_labels, _ = load_images_from_folder(test_folder)

y_train = np.array(train_labels)
y_test = np.array(test_labels)

In [7]:
print(f"Nhãn các class: {class_names}")
print(f"Tổng số ảnh trong tập train: {len(train_images)}")
print(f"Tổng số ảnh trong tập test: {len(test_images)}")

Nhãn các class: ['Cuc', 'Dao', 'Lan', 'Mai', 'Tho']
Tổng số ảnh trong tập train: 150
Tổng số ảnh trong tập test: 50


## Các hàm rút trích đặc trưng

In [8]:
#Histogram với 3 thành phần màu HSV tương ứng 8x8x8
def compute_histogram_feature_hsv(image, bins=8):
    image = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
    hist = cv2.calcHist([image], [0, 1, 2], None, [bins, bins, bins], [0, 180, 0, 256, 0, 256])
    hist = cv2.normalize(hist, hist).flatten()
    return np.array(hist)

In [9]:
X_train_hsv = np.array([compute_histogram_feature_hsv(img,16) for img in train_images])
X_test_hsv = np.array([compute_histogram_feature_hsv(img,16) for img in test_images])

In [10]:
# Định nghĩa tham số cho GridSearchCV
param_grid = {
    'penalty': ['l1', 'l2', 'elasticnet', None],
    'solver': ['lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga']
}

logistic = LogisticRegression(max_iter=50)

grid_search = GridSearchCV(logistic, param_grid, scoring='f1_macro', cv=5, n_jobs=-1)
#grid_search = RandomizedSearchCV(logistic, param_grid, scoring='f1_macro',  n_jobs=-1)
grid_search.fit(X_train_hsv, y_train)


/usr/local/lib/python3.11/dist-packages/sklearn/model_selection/_validation.py:528: FitFailedWarning: 
55 fits failed out of a total of 120.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
5 fits failed with the following error:
Traceback (most recent call last):
  File "/usr/local/lib/python3.11/dist-packages/sklearn/model_selection/_validation.py", line 866, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "/usr/local/lib/python3.11/dist-packages/sklearn/base.py", line 1389, in wrapper
    return fit_method(estimator, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_logistic.py", line 1193, in fit
    solver = _check_solver

GridSearchCV(cv=5, estimator=LogisticRegression(max_iter=50), n_jobs=-1,
             param_grid={'penalty': ['l1', 'l2', 'elasticnet', None],
                         'solver': ['lbfgs', 'liblinear', 'newton-cg',
                                    'newton-cholesky', 'sag', 'saga']},
             scoring='f1_macro')

In [11]:
print("Best Parameters:",  grid_search.best_params_)
print("Best Macro F1-Score:",grid_search.best_score_)

Best Parameters: {'penalty': None, 'solver': 'saga'}
Best Macro F1-Score: 0.8170216450216451


In [12]:
best_model = grid_search.best_estimator_
y_pred_hsv = best_model.predict(X_test_hsv)

test_f1_macro = f1_score(y_test, y_pred_hsv, average='macro')
print(f"Test Macro F1-Score: {test_f1_macro:.4f}")

print(f"Classification Report:")
print(classification_report(y_test, y_pred_hsv))

Test Macro F1-Score: 0.8796
Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.90      0.95        10
           1       0.83      1.00      0.91        10
           2       0.89      0.80      0.84        10
           3       0.82      0.90      0.86        10
           4       0.89      0.80      0.84        10

    accuracy                           0.88        50
   macro avg       0.89      0.88      0.88        50
weighted avg       0.89      0.88      0.88        50

